
# 1. The Data

This notebook contains all the data used for our November 2 report, including the EDGAR filings, CRSP matching, earnings release texts, and three independent sources of release timestamps.

The notebook can be run from top to bottom. The data acquisition scripts are saved to disk by the cells containing them, so they can be reviewed and executed separately. However, they are disabled by default and only run if `RUN_FROM_SCRATCH` is enabled, since they require WRDS access and approximately one day to download the data.



In [ ]:
#setup
!pip install -q "pandas>=2.0"

import glob, os, zipfile
import pandas as pd

#the acquisition pass is off. It needs a WRDS account and about a day of fetching from EDGAR, and it is what produced the files in nov2_data.
RUN_FROM_SCRATCH = False
WRDS_USER = ""

print("pandas", pd.__version__)

pandas 2.2.3


In [ ]:
import os, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Optional private data root; code always comes from this checkout.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
os.environ["TEAM_DIR"] = str(TEAM_DIR)
SCRIPTS = REPO_ROOT / "scripts"
NOV2_DATA = TEAM_DIR / "1_2nov" / "nov2_data"
if not NOV2_DATA.is_dir():
    NOV2_DATA = TEAM_DIR / "files csv"

#data

import os

DATA = str(NOV2_DATA)

NEED = [
    "attrition_funnel.csv",
    "raw_counts.csv",
    "matched_sample.csv",
    "extraction_report.csv",
    "timing_validation.csv",
    "wire_times.csv",
    "three_source_timing.csv",
    "event_table.csv"
]

#we check data files

Path(DATA).mkdir(parents=True, exist_ok=True)

missing = []

for f in NEED:
    path = os.path.join(DATA, f)
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        missing.append(f)

if missing and not RUN_FROM_SCRATCH:
    raise FileNotFoundError(f"Missing files in {DATA}: {missing}. Restore the private data or set RUN_FROM_SCRATCH=True.")

RESULTS = str(TEAM_DIR / 'results/1_results')
MANUAL = str(TEAM_DIR / '2_EDGAR text/Manual validation')

os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MANUAL, exist_ok=True)

for f in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, f)
    if os.path.isfile(path):
        print(f"{os.path.getsize(path):>12,}  {f}")

print("\nresults ->", RESULTS)
print("manual  ->", MANUAL)

Mounted at /content/drive
      37,597  acceptance_header_check.csv
         311  attrition_funnel.csv
  92,679,435  event_table.csv
   1,874,799  extraction_report.csv
   2,983,458  matched_sample.csv
         550  raw_counts.csv
  12,312,462  three_source_timing.csv
   8,382,896  timing_validation.csv
  12,012,726  wire_times.csv

results -> /content/drive/MyDrive/S08 NLP/results/1_results
manual  -> /content/drive/MyDrive/S08 NLP/results/2_EDGAR text/Manual validation


In [ ]:
# Pipeline implementation: scripts/funnel_counts.py
# Run the acquisition cell below to use the checked-in script.


Writing funnel_counts.py


In [ ]:
# Pipeline implementation: scripts/build_event_table.py
# Run the acquisition cell below to use the checked-in script.


Writing build_event_table.py


In [ ]:
# Pipeline implementation: scripts/fetch_acceptance.py
# Run the acquisition cell below to use the checked-in script.


Writing fetch_acceptance.py


In [ ]:
# Pipeline implementation: scripts/wrds_match_and_funnel.py
# Run the acquisition cell below to use the checked-in script.


Writing wrds_match_and_funnel.py


In [ ]:
# Pipeline implementation: scripts/fetch_releases.py
# Run the acquisition cell below to use the checked-in script.


Writing fetch_releases.py


In [ ]:
# Pipeline implementation: scripts/validate_timing.py
# Run the acquisition cell below to use the checked-in script.


Writing validate_timing.py


In [ ]:
# Pipeline implementation: scripts/wire_times.py
# Run the acquisition cell below to use the checked-in script.


Writing wire_times.py


In [ ]:
# Pipeline implementation: scripts/three_source_timing.py
# Run the acquisition cell below to use the checked-in script.


Writing three_source_timing.py


In [ ]:
# Pipeline implementation: scripts/check_access.py
# Run the acquisition cell below to use the checked-in script.


Writing check_access.py


In [ ]:
#acquisition pass
import subprocess, sys

PIPELINE = [
    'funnel_counts.py',
    'build_event_table.py',
    'fetch_acceptance.py',
    'wrds_match_and_funnel.py',
    'fetch_releases.py',
    'validate_timing.py',
    'wire_times.py',
    'three_source_timing.py',
    'check_access.py'
]

ZIP = str(Path(DATA) / "submissions.zip")

if RUN_FROM_SCRATCH:
    for script in PIPELINE:
        print("=" * 60, script, flush=True)
        if script == "funnel_counts.py":
            subprocess.run([sys.executable, str(SCRIPTS / script),
                            "--zip", ZIP], check=True)
        elif script == "build_event_table.py":
            subprocess.run([
                sys.executable, str(SCRIPTS / script),
                "--zip", ZIP,
                "--out", os.path.join(DATA, "event_table.csv"),
                "--acceptance",
                str(TEAM_DIR / 'files csv/acceptance_times.csv')
            ], check=True)
        else:
            subprocess.run([sys.executable, str(SCRIPTS / script)], check=True)

else:
    for script in PIPELINE:
        print("written, not run:", script)

written, not run: funnel_counts.py
written, not run: build_event_table.py
written, not run: fetch_acceptance.py
written, not run: wrds_match_and_funnel.py
written, not run: fetch_releases.py
written, not run: validate_timing.py
written, not run: wire_times.py
written, not run: three_source_timing.py
written, not run: check_access.py


In [ ]:
ok, bad = 0, []


def check(label, got, want, pct=None):
    global ok
    hit = (round(got, 4) == round(want, 4))
    if hit:
        ok += 1
    else:
        bad.append(f"{label}: got {got}, report says {want}")
    tail = f"  ({pct:.1f}%)" if pct is not None else ""
    print(f"  {'ok ' if hit else 'NO '}{label:<52}{got:>12,}{tail}")

In [ ]:
print("1. raw counts")
rc = pd.read_csv(f"{DATA}/raw_counts.csv")
#the file carries its own otal row; the yearly rows are the data.
total = rc[rc.year == "TOTAL"].iloc[0]
rc = rc[rc.year != "TOTAL"].astype({"year": int})
check("TOTAL row agrees with the sum of the years", int(rc.item_2_02_filings.sum()), int(total.item_2_02_filings))
check("Item 2.02 filings, 2006-2025", int(rc.item_2_02_filings.sum()), 383153)
check("all 8-K filings", int(rc.all_8k_filings.sum()), 1549567)
check("fewest distinct filers in a year", int(rc.distinct_filers_with_2_02.min()), 4277)
check("most distinct filers in a year", int(rc.distinct_filers_with_2_02.max()), 6003)
check("years covered", int(rc.year.nunique()), 20)
first, last = rc.set_index("year").item_2_02_filings.loc[[2006, 2025]]
check("fall in Item 2.02 count, 2006 to 2025, in percent", round(100 * (1 - last / first)), 27)

1. raw counts
  ok TOTAL row agrees with the sum of the years               383,153
  ok Item 2.02 filings, 2006-2025                             383,153
  ok all 8-K filings                                        1,549,567
  ok fewest distinct filers in a year                           4,277
  ok most distinct filers in a year                             6,003
  ok years covered                                                 20
  ok fall in Item 2.02 count, 2006 to 2025, in percent             27


In [ ]:
print("\n2. match attrition")
fn = pd.read_csv(f"{DATA}/attrition_funnel.csv")
ms = pd.read_csv(f"{DATA}/matched_sample.csv")
check("events after the last stage", int(fn.remaining.iloc[-1]), 117926)
check("matched sample rows", len(ms), 117926)
rem = fn.remaining.to_numpy()
check("stages where removed equals the drop in remaining",
      int(sum(int(fn.removed.iloc[i]) == rem[i - 1] - rem[i]
              for i in range(1, len(fn)))), len(fn) - 1)
check("largest single removal, the size cut", int(fn.removed.max()), 79505)
check("median market capitalisation, $bn",
      round(ms.mktcap.median() / 1e6, 1), 3.4)
check("median number of analysts", int(ms.n_analysts.median()), 9)
check("events in 2025, lost with the legacy CRSP file",
      int((pd.to_datetime(ms.session_date).dt.year == 2025).sum()), 5791)


2. match attrition
  NO events after the last stage                              117,836
  ok matched sample rows                                      117,926
  ok stages where removed equals the drop in remaining              6
  NO largest single removal, the size cut                      79,685
  ok median market capitalisation, $bn                            3.4
  ok median number of analysts                                      9
  ok events in 2025, lost with the legacy CRSP file             5,791


In [ ]:
print("\n3. release text")
ex = pd.read_csv(f"{DATA}/extraction_report.csv")
src = ex.source.fillna("")
got = src.ne("none") & src.ne("index fetch failed")
check("events with a release recovered", int(got.sum()), 116983,
      100 * got.mean())
v1 = src.str.startswith("EX-99.1")
other = src.str.startswith("EX-99") & ~v1
check("EX-99.1 and its variants", int(v1.sum()), 103852, 100 * v1.mean())
check("other EX-99 exhibits", int(other.sum()), 11269, 100 * other.mean())
body = src.eq("8-K body")
check("the 8-K body itself", int(body.sum()), 1862, 100 * body.mean())
check("no prose document found", int(src.eq("none").sum()), 941,
      100 * src.eq("none").mean())
check("index page unreachable", int(src.eq("index fetch failed").sum()), 2)
check("share lost by taking EX-99.1 alone, in percent",
      round(100 * (got.sum() - v1.sum()) / got.sum(), 1), 11.2)
w = ex.loc[got, "words"]
check("median release, words", int(w.median()), 3157)
check("tenth percentile, words", int(w.quantile(0.10)), 1437)
check("ninetieth percentile, words", round(w.quantile(0.90)), 5960)
check("median tables dropped", int(ex.loc[got, "tables_dropped"].median()), 3)


3. release text
  ok events with a release recovered                          116,983  (99.2%)
  ok EX-99.1 and its variants                                 103,852  (88.1%)
  ok other EX-99 exhibits                                      11,269  (9.6%)
  ok the 8-K body itself                                        1,862  (1.6%)
  ok no prose document found                                      941  (0.8%)
  ok index page unreachable                                         2
  ok share lost by taking EX-99.1 alone, in percent              11.2
  ok median release, words                                      3,157
  ok tenth percentile, words                                    1,437
  ok ninetieth percentile, words                                5,960
  ok median tables dropped                                          3


In [ ]:
print("\n4. release timing")
CLOSE_S, OPEN_S = 16 * 3600, 9 * 3600 + 1800


def session(dates, secs):
    d = pd.to_datetime(dates, errors="coerce")
    missing = secs.isna() | (secs == 0)
    after = ~missing & (secs >= CLOSE_S)
    s = d.dt.date.astype("string")
    s = s.where(~after, (d + pd.Timedelta(days=1)).dt.date.astype("string"))
    return s.where(d.notna() & ~missing)


def secs_of(ts):
    t = pd.to_datetime(ts, errors="coerce")
    return t.dt.hour * 3600 + t.dt.minute * 60 + t.dt.second


t = pd.read_csv(f"{DATA}/timing_validation.csv")
wi = pd.read_csv(f"{DATA}/wire_times.csv")
n = len(t)
check("firm-days with a Dow Jones wire time", len(wi), 444915)
check("distinct securities on the wire file", int(wi.permno.nunique()), 8853)

has_i_all = t.ibes_flag.notna()
check("events with an I/B/E/S announcement", int(has_i_all.sum()), 106418,
      100 * has_i_all.mean())
check("EDGAR and I/B/E/S on the same session", int(t.agrees.sum()), 88448,
      100 * t.agrees.sum() / has_i_all.sum())
check("kept if two sources must agree, in percent",
      round(100 * t.agrees.mean(), 1), 75.0)
check("dropped by that rule", int(n - t.agrees.sum()), 29478)
check("  of those, no I/B/E/S match", int((~has_i_all).sum()), 11508)
dis = (~t.agrees) & has_i_all
check("  of those, matched to a different session", int(dis.sum()), 17970)

d = t[dis].copy()
d["lag"] = (pd.to_datetime(d.session_date)
            - pd.to_datetime(d.anndats)).dt.days
same = d.lag == 0
check("same date, I/B/E/S after the close, EDGAR in session",
      int((same & d.when.eq("intraday")).sum()), 12809)
check("same date, I/B/E/S after the close, EDGAR pre-open",
      int((same & d.when.eq("pre_open")).sum()), 154)
check("same date, both after the close, different session",
      int((same & d.when.eq("after_close")).sum()), 27)
check("same date, differing only on the side of the close",
      int(same.sum()), 12990)
check("EDGAR session one to three days after the I/B/E/S date",
      int(d.lag.between(1, 3).sum()), 4706)
check("EDGAR session before the I/B/E/S date", int((d.lag < 0).sum()), 274)

t["permno"] = pd.to_numeric(t.permno, errors="coerce").astype("Int64")
wi["permno"] = pd.to_numeric(wi.permno, errors="coerce").astype("Int64")
wi["wire_dt"] = pd.to_datetime(wi.wire_time_et, errors="coerce")
t["edgar_dt"] = pd.to_datetime(t.session_date, errors="coerce")
m = pd.merge_asof(t.sort_values("edgar_dt"),
                  wi.sort_values("wire_dt").dropna(subset=["wire_dt"])
                    [["permno", "wire_dt"]],
                  left_on="edgar_dt", right_on="wire_dt", by="permno",
                  direction="nearest", tolerance=pd.Timedelta("3D"))
m["wire_session"] = session(m.wire_dt, secs_of(m.wire_dt))
has_w, has_i = m.wire_session.notna(), m.ibes_session.notna()
check("events given a wire time", int(has_w.sum()), 90437, 100 * has_w.mean())

ei = has_i & (m.session_date == m.ibes_session)
ew = has_w & (m.session_date == m.wire_session)
iw = has_i & has_w & (m.ibes_session == m.wire_session)
check("I/B/E/S agrees with the wire", int(iw.sum()), 86374,
      100 * iw.sum() / (has_i & has_w).sum())
check("  out of events where both exist", int((has_i & has_w).sum()), 87647)
check("EDGAR agrees with the wire", int(ew.sum()), 75775,
      100 * ew.sum() / has_w.sum())
check("EDGAR agrees with I/B/E/S", int(ei.sum()), 88448,
      100 * ei.sum() / has_i.sum())

two = (ei.astype(int) + ew.astype(int) + iw.astype(int)) >= 1
check("retained when two of three sources agree", int(two.sum()), 104259,
      100 * two.mean())
check("dropped by that rule", int(n - two.sum()), 13667)
only_e = ~has_i & ~has_w
check("  of those, with no second source", int(only_e.sum()), 8718)
check("  of those, with a second source that disagrees",
      int(n - two.sum() - only_e.sum()), 4949)

m["session_final"] = m.wire_session.where(has_w, m.session_date)
moved = m.session_final != m.session_date
check("events the wire moves to another session", int(moved.sum()), 14662,
      100 * moved.mean())
delta = (pd.to_datetime(m.session_final, errors="coerce")
         - pd.to_datetime(m.session_date, errors="coerce")).dt.days
check("  EDGAR one to three sessions too early", int((delta > 0).sum()), 10526)
check("  EDGAR one to three sessions too late", int((delta < 0).sum()), 4136)


4. release timing
  ok firm-days with a Dow Jones wire time                     444,915
  ok distinct securities on the wire file                       8,853
  ok events with an I/B/E/S announcement                      106,418  (90.2%)
  ok EDGAR and I/B/E/S on the same session                     88,448  (83.1%)
  ok kept if two sources must agree, in percent                  75.0
  ok dropped by that rule                                      29,478
  ok   of those, no I/B/E/S match                              11,508
  ok   of those, matched to a different session                17,970
  ok same date, I/B/E/S after the close, EDGAR in session      12,809
  ok same date, I/B/E/S after the close, EDGAR pre-open           154
  ok same date, both after the close, different session            27
  ok same date, differing only on the side of the close        12,990
  ok EDGAR session one to three days after the I/B/E/S date       4,706
  ok EDGAR session before the I/B/E/S date         

In [ ]:
chk = pd.read_csv(f"{DATA}/acceptance_header_check.csv")
got = chk.dropna(subset=["delta_hours"])
same = (got.delta_hours == 0)

print("EDGAR acceptance time: the submissions JSON against the filing header")
print(f"  filings checked                 {len(got):>6,}")
print(f"  JSON agrees with the header     {same.sum():>6,}  "
      f"{100 * same.mean():5.1f}%")
print(f"  JSON disagrees                  {(~same).sum():>6,}  "
      f"{100 * (~same).mean():5.1f}%")
print("\n  what the field actually holds, decided filing by filing")
for name, n in chk.regime.value_counts().items():
    print(f"    {n:>5,}  {name}")
print("""
    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter""")
print("  Three regimes mixed inside the same year, so no rule recovers the")
print("  truth from the JSON alone. fetch_acceptance.py reads the header.")

print("\nStill to rerun, in this order, once the headers are fetched:")
for s in ["fetch_acceptance.py", "build_event_table.py",
          "wrds_match_and_funnel.py", "validate_timing.py",
          "three_source_timing.py"]:
    print("   ", s)
print("\nThe release texts do not need refetching: the extraction does not")
print("depend on the timestamp. The CRSP windows do.")

EDGAR acceptance time: the submissions JSON against the filing header
  filings checked                    300
  JSON agrees with the header         99   33.0%
  JSON disagrees                     201   67.0%

  what the field actually holds, decided filing by filing
      128  Eastern labelled Z
       99  true UTC
       73  converted to UTC twice

    true UTC                 the field is what it claims to be
    Eastern labelled Z       the clock reading is Eastern and the Z is a
                             lie, so converting it moves the release four
                             or five hours too early
    converted to UTC twice   the offset was added on top of a value that
                             already carried it, which lands eight hours
                             past Eastern in summer and ten in winter
  Three regimes mixed inside the same year, so no rule recovers the
  truth from the JSON alone. fetch_acceptance.py reads the header.

Still to rerun, in this order, o

In [ ]:
ts = pd.read_csv(f"{DATA}/three_source_timing.csv",
                 dtype={"accession": str})
conflict = ts[ts.wire_session.notna() & ts.ibes_session_c.notna()
              & ((ts.edgar_session != ts.wire_session)
                 | (ts.edgar_session != ts.ibes_session_c))]

hand = conflict.sample(min(50, len(conflict)), random_state=20261104).copy()
hand["edgar_url"] = [
    "https://www.sec.gov/Archives/edgar/data/"
    + f"{a.replace('-', '')}/{a}-index.htm" for a in hand.accession]
for c in ["session_you_read", "source_you_trust", "note"]:
    hand[c] = ""

path = os.path.join(MANUAL, "timing_handcheck_50.csv")
hand.to_csv(path, index=False, encoding="utf-8-sig")
print(f"{len(conflict):,} events where the sources disagree, "
      f"{len(hand)} drawn")
print("->", path)
print()
print("For each row: open the filing on EDGAR, read the acceptance time in")
print("the header, and say which session the release really belongs to.")
print("`source_you_trust` is edgar, ibes or wire. The point is not the")
print("fifty verdicts, it is whether one source is wrong in one direction.")

14,317 events where the sources disagree, 50 drawn
-> /content/drive/MyDrive/S08 NLP/results/2_EDGAR text/Manual validation/timing_handcheck_50.csv

For each row: open the filing on EDGAR, read the acceptance time in
the header, and say which session the release really belongs to.
`source_you_trust` is edgar, ibes or wire. The point is not the
fifty verdicts, it is whether one source is wrong in one direction.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fn = pd.read_csv(f"{DATA}/attrition_funnel.csv")
rc = pd.read_csv(f"{DATA}/raw_counts.csv")
ms = pd.read_csv(f"{DATA}/matched_sample.csv",
                 usecols=["session_date"])
ts = pd.read_csv(f"{DATA}/three_source_timing.csv",
                 usecols=["edgar_session", "wire_session", "ibes_session_c"])

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

a = ax[0][0]
a.barh(fn.step[::-1], fn.remaining[::-1], color="#4a5568")
a.set_title("Attrition funnel")
a.set_xlabel("filings remaining")
for i, n in enumerate(fn.remaining[::-1]):
    a.text(n, i, f" {n:,}", va="center", fontsize=8)
a.tick_params(axis="y", labelsize=8)

a = ax[0][1]
y = rc[rc.year != "TOTAL"].copy()
y["year"] = y.year.astype(int)
a.plot(y.year, y.item_2_02_filings, marker="o", color="#2c5282",
       label="Item 2.02 filings")
ev_per_year = (pd.to_datetime(ms.session_date, errors="coerce")
                 .dt.year.value_counts().sort_index())
a.plot(ev_per_year.index, ev_per_year.values, marker="o", color="#9b2c2c",
       label="in the final sample")
a.set_title("Filings a year, before and after the filters")
a.legend(frameon=False)

a = ax[1][0]
#each rate is over the events where both sources speak. Dividing by every event instead would mix "the two disagree" with "one is silent", which are different facts and move the number by ten points.
def agree(a, b):
    both = ts[a].notna() & ts[b].notna()
    return (ts.loc[both, a] == ts.loc[both, b]).mean(), int(both.sum())


pairs = {f"EDGAR = I/B/E/S\nn={agree('edgar_session', 'ibes_session_c')[1]:,}":
         agree("edgar_session", "ibes_session_c")[0],
         f"EDGAR = wire\nn={agree('edgar_session', 'wire_session')[1]:,}":
         agree("edgar_session", "wire_session")[0],
         f"I/B/E/S = wire\nn={agree('ibes_session_c', 'wire_session')[1]:,}":
         agree("ibes_session_c", "wire_session")[0]}
a.bar(list(pairs), [100 * v for v in pairs.values()], color="#4a5568")
a.set_ylim(0, 100)
a.set_ylabel("% of events in the same session")
a.set_title("Do two sources put the release in the same session?")
for i, v in enumerate(pairs.values()):
    a.text(i, 100 * v, f"{100 * v:.1f}%", ha="center", va="bottom",
           fontsize=9)

a = ax[1][1]
delta = (pd.to_datetime(ts.wire_session, errors="coerce")
         - pd.to_datetime(ts.edgar_session, errors="coerce")).dt.days
d = delta[(delta.notna()) & (delta != 0) & (delta.abs() <= 5)]
a.bar(d.value_counts().sort_index().index,
      d.value_counts().sort_index().values, color="#9b2c2c")
a.set_title("How far the wire moves the session")
a.set_xlabel("sessions, wire minus EDGAR")

for row in ax:
    for a in row:
        a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
#access

QUERIES = {
    "crsp.dsf_v2": "select max(dlycaldt) from crsp.dsf_v2",
    "crsp.dsf": "select max(date) from crsp.dsf",
    "ibes.det_guidance": "select count(*) from ibes.det_guidance "
                         "where anndats >= '2024-01-01'",
    "ibes.actu_epsus": "select max(anndats) from ibes.actu_epsus",
    "ibes.detu_epsus": "select max(anndats) from ibes.detu_epsus",
    "wrdsapps.ibcrsphist": "select count(*) from wrdsapps.ibcrsphist "
                           "where score <= 3",
    "crsp.ccmxpf_lnkhist": "select count(*) from crsp.ccmxpf_lnkhist",
    "wrdssec.wciklink_gvkey": "select count(*) from wrdssec.wciklink_gvkey",
}

if RUN_FROM_SCRATCH:
    !pip install -q wrds
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USER)
    for table, q in QUERIES.items():
        try:
            print(f"{table:<26}{db.raw_sql(q).iloc[0, 0]}")
        except Exception as exc:
            print(f"{table:<26}refused: {type(exc).__name__}")
    db.close()
else:
    for table, q in QUERIES.items():
        print(f"{table:<26}{q}")

crsp.dsf_v2               select max(dlycaldt) from crsp.dsf_v2
crsp.dsf                  select max(date) from crsp.dsf
ibes.det_guidance         select count(*) from ibes.det_guidance where anndats >= '2024-01-01'
ibes.actu_epsus           select max(anndats) from ibes.actu_epsus
ibes.detu_epsus           select max(anndats) from ibes.detu_epsus
wrdsapps.ibcrsphist       select count(*) from wrdsapps.ibcrsphist where score <= 3
crsp.ccmxpf_lnkhist       select count(*) from crsp.ccmxpf_lnkhist
wrdssec.wciklink_gvkey    select count(*) from wrdssec.wciklink_gvkey
